# Predicting asset failure within 90 days

**Question:** Can information available at inspection time predict `failed_90d`?

This notebook treats the task as an imbalanced binary-classification problem. It emphasizes:

- feature availability and target leakage;
- chronological validation with a 90-day embargo;
- ranking metrics suitable for rare events (PR-AUC and lift), alongside ROC-AUC and calibration;
- comparison with simple baselines and a deliberately leaky sensitivity model.

The notebook expects `data.csv` and `columns.csv` in the same directory. It is deterministic (`random_state=42`) and does not modify the source data.

In [ ]:
from pathlib import Path
import os
import warnings

os.environ.setdefault("LOKY_MAX_CPU_COUNT", "2")
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display, Markdown
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score, brier_score_loss, precision_recall_curve,
    roc_auc_score, roc_curve
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
pd.set_option("display.max_colwidth", 100)
plt.style.use("seaborn-v0_8-whitegrid")

## 1. Load and validate

In [ ]:
DATA_PATH = Path("data.csv")
DICT_PATH = Path("columns.csv")

assert DATA_PATH.exists() and DICT_PATH.exists(), "Place data.csv and columns.csv beside this notebook."

dictionary = pd.read_csv(DICT_PATH)
df = pd.read_csv(DATA_PATH, parse_dates=["inspected_at"])

expected = set(dictionary["column"])
assert expected == set(df.columns), f"Schema mismatch: {expected.symmetric_difference(df.columns)}"
assert df["failed_90d"].isin([0, 1]).all(), "Target must be binary."
assert df["inspected_at"].notna().all(), "Inspection dates must parse."

display(dictionary)
display(df.head())

In [ ]:
audit = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing": df.isna().sum(),
    "unique": df.nunique(dropna=False),
})
summary = pd.Series({
    "rows": len(df),
    "failure_count": int(df["failed_90d"].sum()),
    "failure_rate": df["failed_90d"].mean(),
    "date_min": df["inspected_at"].min(),
    "date_max": df["inspected_at"].max(),
    "duplicate_rows": int(df.duplicated().sum()),
    "duplicate_asset_timestamp": int(df.duplicated(["asset_tag", "inspected_at"]).sum()),
})
display(summary.to_frame("value"))
display(audit)

## 2. Data quality, drift, and leakage audit

`asset_tag` is an identifier, and every value is unique here, so it cannot teach the model a repeatable asset effect. It is excluded.

`parts_replaced` is described as the count on the work order that **closed** the inspection record. Unless that value is already known at the exact prediction moment, it is post-inspection information. The primary models therefore exclude it. A separate sensitivity model includes it only to demonstrate the effect of leakage.

In [ ]:
print("Asset rows per tag:")
display(df.groupby("asset_tag").size().describe().to_frame("inspection_count"))

print("Failure rate conditional on parts_replaced:")
display(pd.crosstab(df["parts_replaced"], df["failed_90d"], margins=True).join(
    df.groupby("parts_replaced")["failed_90d"].mean().rename("failure_rate"), how="left"
))

monthly = (df.set_index("inspected_at")["failed_90d"]
             .resample("MS").agg(["size", "sum", "mean"])
             .rename(columns={"size": "inspections", "sum": "failures", "mean": "failure_rate"}))
ax = monthly["failure_rate"].plot(figsize=(10, 3), marker="o", title="Monthly 90-day failure-label rate")
ax.set_ylabel("Failure rate")
plt.show()
display(monthly.tail(12))

In [ ]:
numeric = ["vibration_rms", "bearing_temp_c", "ambient_temp_c", "meter_hours", "load_pct"]
group_summary = df.groupby("failed_90d")[numeric + ["parts_replaced"]].mean().T
group_summary.columns = ["no_failure_mean", "failure_mean"]
group_summary["standardized_difference"] = (
    group_summary["failure_mean"] - group_summary["no_failure_mean"]
) / df[numeric + ["parts_replaced"]].std()
display(group_summary.sort_values("standardized_difference", key=abs, ascending=False))

## 3. Honest temporal holdout

A random split would let later inspections inform predictions for earlier ones and would hide temporal drift. The final 20% of inspections by time is held out. Training ends **90 days before** the first test inspection, matching the outcome horizon and avoiding overlapping label windows. Rows in the embargo are intentionally unused.

The timestamp is known at inspection time. Calendar features are included, but a no-calendar sensitivity model checks whether they dominate the result.

In [ ]:
df = df.sort_values("inspected_at").reset_index(drop=True)
test_start = df["inspected_at"].quantile(0.80)
train_end = test_start - pd.Timedelta(days=90)

train = df.loc[df["inspected_at"] < train_end].copy()
embargo = df.loc[(df["inspected_at"] >= train_end) & (df["inspected_at"] < test_start)].copy()
test = df.loc[df["inspected_at"] >= test_start].copy()

split_summary = pd.DataFrame({
    "rows": [len(train), len(embargo), len(test)],
    "failures": [train.failed_90d.sum(), embargo.failed_90d.sum(), test.failed_90d.sum()],
    "failure_rate": [train.failed_90d.mean(), embargo.failed_90d.mean(), test.failed_90d.mean()],
    "start": [train.inspected_at.min(), embargo.inspected_at.min(), test.inspected_at.min()],
    "end": [train.inspected_at.max(), embargo.inspected_at.max(), test.inspected_at.max()],
}, index=["train", "embargo (unused)", "test"])
display(split_summary)

In [ ]:
date_origin = df["inspected_at"].min()

def add_time_features(frame):
    out = frame.copy()
    month = out["inspected_at"].dt.month
    out["inspection_day_index"] = (out["inspected_at"] - date_origin).dt.total_seconds() / 86400
    out["month_sin"] = np.sin(2 * np.pi * month / 12)
    out["month_cos"] = np.cos(2 * np.pi * month / 12)
    return out

train_m = add_time_features(train)
test_m = add_time_features(test)

physical_features = ["site", "vibration_rms", "bearing_temp_c", "ambient_temp_c", "meter_hours", "load_pct"]
calendar_features = ["inspection_day_index", "month_sin", "month_cos"]
operational_features = physical_features + calendar_features

def make_preprocessor(features, dense=False):
    categorical = [c for c in features if c == "site"]
    numerical = [c for c in features if c != "site"]
    return ColumnTransformer([
        ("num", Pipeline([
            ("impute", SimpleImputer(strategy="median")),
            ("scale", StandardScaler()),
        ]), numerical),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=not dense), categorical),
    ])

def make_pipeline(estimator, features, dense=False):
    return Pipeline([
        ("prepare", make_preprocessor(features, dense=dense)),
        ("model", estimator),
    ])

## 4. Fit baselines and candidate models

In [ ]:
models = {
    "constant prior": make_pipeline(DummyClassifier(strategy="prior"), operational_features),
    "logistic": make_pipeline(LogisticRegression(max_iter=2000), operational_features),
    "random forest": make_pipeline(RandomForestClassifier(
        n_estimators=500, min_samples_leaf=15, max_features="sqrt",
        class_weight="balanced_subsample", n_jobs=-1, random_state=RANDOM_STATE
    ), operational_features),
    "hist gradient boost": make_pipeline(HistGradientBoostingClassifier(
        max_iter=250, learning_rate=0.05, max_leaf_nodes=15,
        l2_regularization=1.0, random_state=RANDOM_STATE
    ), operational_features, dense=True),
    "HGB, no calendar": make_pipeline(HistGradientBoostingClassifier(
        max_iter=250, learning_rate=0.05, max_leaf_nodes=15,
        l2_regularization=1.0, random_state=RANDOM_STATE
    ), physical_features, dense=True),
}

y_train = train_m["failed_90d"]
y_test = test_m["failed_90d"].to_numpy()
predictions = {}

for name, model in models.items():
    features = physical_features if name == "HGB, no calendar" else operational_features
    model.fit(train_m[features], y_train)
    predictions[name] = model.predict_proba(test_m[features])[:, 1]

def metric_row(y, probability, alert_fraction=0.10):
    n_alert = max(1, int(np.ceil(len(y) * alert_fraction)))
    chosen = np.argsort(-probability)[:n_alert]
    precision_at_budget = y[chosen].mean()
    recall_at_budget = y[chosen].sum() / y.sum()
    prevalence = y.mean()
    return {
        "ROC-AUC": roc_auc_score(y, probability),
        "PR-AUC": average_precision_score(y, probability),
        "Brier (lower better)": brier_score_loss(y, probability),
        "precision@10%": precision_at_budget,
        "recall@10%": recall_at_budget,
        "lift@10%": precision_at_budget / prevalence,
    }

results = pd.DataFrame({name: metric_row(y_test, p) for name, p in predictions.items()}).T
display(results.sort_values("PR-AUC", ascending=False).style.format("{:.3f}"))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for name, p in predictions.items():
    if name == "constant prior":
        continue
    fpr, tpr, _ = roc_curve(y_test, p)
    precision, recall, _ = precision_recall_curve(y_test, p)
    axes[0].plot(fpr, tpr, label=f"{name} ({roc_auc_score(y_test, p):.3f})")
    axes[1].plot(recall, precision, label=f"{name} ({average_precision_score(y_test, p):.3f})")
axes[0].plot([0, 1], [0, 1], "--", color="gray")
axes[1].axhline(y_test.mean(), ls="--", color="gray", label=f"prevalence ({y_test.mean():.3f})")
axes[0].set(title="ROC curve", xlabel="False-positive rate", ylabel="True-positive rate")
axes[1].set(title="Precision-recall curve", xlabel="Recall", ylabel="Precision")
for ax in axes:
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

### Uncertainty on the final holdout

The latest holdout has relatively few failures, so point estimates can look more certain than they are. The stratified bootstrap below preserves the positive/negative counts and estimates 95% intervals for the leading operational model.

In [ ]:
best_name = "hist gradient boost"
best_p = predictions[best_name]

def stratified_bootstrap(y, p, repeats=2000, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed)
    pos = np.flatnonzero(y == 1)
    neg = np.flatnonzero(y == 0)
    records = []
    for _ in range(repeats):
        idx = np.concatenate([rng.choice(pos, len(pos), replace=True), rng.choice(neg, len(neg), replace=True)])
        records.append(metric_row(y[idx], p[idx]))
    return pd.DataFrame(records)

boot = stratified_bootstrap(y_test, best_p)
intervals = pd.DataFrame({
    "estimate": pd.Series(metric_row(y_test, best_p)),
    "2.5%": boot.quantile(0.025),
    "97.5%": boot.quantile(0.975),
})
display(intervals.style.format("{:.3f}"))

## 5. Rolling temporal backtests

In [ ]:
def evaluate_window(test_window_start, window_days=90):
    test_window_start = pd.Timestamp(test_window_start)
    local_train_end = test_window_start - pd.Timedelta(days=90)
    local_test_end = test_window_start + pd.Timedelta(days=window_days)
    tr = add_time_features(df[df.inspected_at < local_train_end])
    va = add_time_features(df[(df.inspected_at >= test_window_start) & (df.inspected_at < local_test_end)])
    if tr.failed_90d.nunique() < 2 or va.failed_90d.nunique() < 2:
        return None
    model = make_pipeline(HistGradientBoostingClassifier(
        max_iter=250, learning_rate=0.05, max_leaf_nodes=15,
        l2_regularization=1.0, random_state=RANDOM_STATE
    ), operational_features, dense=True)
    model.fit(tr[operational_features], tr.failed_90d)
    p = model.predict_proba(va[operational_features])[:, 1]
    metrics = metric_row(va.failed_90d.to_numpy(), p)
    return {
        "test_start": test_window_start,
        "train_rows": len(tr), "test_rows": len(va),
        "test_failures": int(va.failed_90d.sum()), "test_prevalence": va.failed_90d.mean(),
        **metrics,
    }

window_starts = pd.to_datetime(["2025-01-01", "2025-04-01", "2025-07-01", "2025-10-01"])
backtests = pd.DataFrame(filter(None, [evaluate_window(d) for d in window_starts])).set_index("test_start")
display(backtests.style.format({
    "test_prevalence": "{:.3f}", "ROC-AUC": "{:.3f}", "PR-AUC": "{:.3f}",
    "Brier (lower better)": "{:.3f}", "precision@10%": "{:.3f}",
    "recall@10%": "{:.3f}", "lift@10%": "{:.2f}",
}))
display(backtests[["ROC-AUC", "PR-AUC", "lift@10%"]].agg(["mean", "min", "max"]))

## 6. What drives predictions?

In [ ]:
best_model = models[best_name]
importance = permutation_importance(
    best_model, test_m[operational_features], y_test,
    scoring="average_precision", n_repeats=30, random_state=RANDOM_STATE, n_jobs=1
)
importance_df = pd.DataFrame({
    "feature": operational_features,
    "mean_PR_AUC_drop": importance.importances_mean,
    "std": importance.importances_std,
}).sort_values("mean_PR_AUC_drop", ascending=False)
display(importance_df)

ax = importance_df.sort_values("mean_PR_AUC_drop").plot.barh(
    x="feature", y="mean_PR_AUC_drop", xerr="std", legend=False, figsize=(8, 4),
    title="Permutation importance on final holdout"
)
ax.set_xlabel("Decrease in PR-AUC after shuffling")
plt.tight_layout()
plt.show()

## 7. Leakage sensitivity: include `parts_replaced`

This is **not** a deployable model unless the organization confirms that `parts_replaced` is finalized before the score is needed. Its purpose is to show how misleading validation can become when post-inspection information is used.

In [ ]:
leaky_features = operational_features + ["parts_replaced"]
leaky_model = make_pipeline(HistGradientBoostingClassifier(
    max_iter=250, learning_rate=0.05, max_leaf_nodes=15,
    l2_regularization=1.0, random_state=RANDOM_STATE
), leaky_features, dense=True)
leaky_model.fit(train_m[leaky_features], y_train)
leaky_p = leaky_model.predict_proba(test_m[leaky_features])[:, 1]

comparison = pd.DataFrame({
    "operational HGB": metric_row(y_test, best_p),
    "leaky HGB (+ parts_replaced)": metric_row(y_test, leaky_p),
}).T
display(comparison.style.format("{:.3f}"))

## 8. Conclusion

In [ ]:
point = results.loc[best_name]
ci = intervals
bt_auc = backtests["ROC-AUC"].mean()
bt_lift = backtests["lift@10%"].mean()

display(Markdown(f''' 
### Answer: **yes, there is useful ranking signal, but this is not yet production proof**

On the untouched latest-time holdout ({len(test):,} inspections, {int(y_test.sum())} failures), the best non-leaky model achieved:

- **ROC-AUC {point['ROC-AUC']:.3f}** (bootstrap 95% interval {ci.loc['ROC-AUC', '2.5%']:.3f}–{ci.loc['ROC-AUC', '97.5%']:.3f});
- **PR-AUC {point['PR-AUC']:.3f}** versus a {y_test.mean():.3f} prevalence/no-skill reference;
- at a **10% inspection-review budget**, precision was {point['precision@10%']:.3f}, recall was {point['recall@10%']:.3f}, and lift was **{point['lift@10%']:.1f}×** over random selection.

Across rolling 90-day backtests, mean ROC-AUC was {bt_auc:.3f} and mean lift at 10% was {bt_lift:.1f}×. This supports genuine discrimination, though performance varies over time.

### Important limitations

1. **Strong target drift:** the failure rate changes substantially over time (training {y_train.mean():.1%}; final test {y_test.mean():.1%}). Probability calibration therefore needs monitoring or recalibration; ranking metrics are more trustworthy than raw probabilities here.
2. **Small positive test sample:** only {int(y_test.sum())} failures occur in the final holdout, producing wide uncertainty.
3. **Potential leakage:** `parts_replaced` almost reveals the target and was excluded. Confirm the exact timestamps and availability of every source field before deployment.
4. **No repeat inspections per asset:** all 6,000 asset tags are unique, so this analysis cannot establish performance on repeat inspections of known assets or model within-asset history.
5. **Operational threshold not chosen:** the 10% alert budget is illustrative. Select a threshold using the relative cost of missed failures and unnecessary maintenance, then validate prospectively.

### Recommended next step

Run a prospective shadow test on future inspections, logging only values available at score time. Track PR-AUC, recall at the actual maintenance capacity, calibration, and performance by site. If the shadow test preserves the observed lift, the model is a credible candidate for operational use.
'''))